In [1]:
import pandas as pd
pd.options.display.max_columns = None
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, PolynomialFeatures
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier, Perceptron, SGDClassifier, RidgeClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, cross_val_score, train_test_split, HalvingGridSearchCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA

### Preprocessing

In [3]:
df = pd.read_csv('fraudTrain.csv', index_col=0, parse_dates=['trans_date_trans_time'])

Based on discoveries of EDA, I delete some featurs and transform others.

In [6]:
df.rename(columns={'trans_date_trans_time':'time', 'amt':'amount'}, inplace=True)
df['industry_code'] = df['cc_num'].astype(str).apply(lambda x: x[0:1]).astype(int)
df['age'] = (pd.to_datetime('today') - pd.to_datetime(df['dob'])).dt.days/365.25 
df.drop(columns=['first', 'last', 'street', 'trans_num', 'unix_time', 'cc_num', 'dob', 'lat', 'long', 'merch_lat', 'merch_long', 'zip', 'city', 'job', 'merchant'], inplace=True)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns='is_fraud'), df['is_fraud'], test_size=0.25, random_state=1)

In [8]:
X_train['time'] = (X_train['time'] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')

X_train_num = X_train.drop(columns=['category', 'gender', 'state'])
X_train_cat = X_train[['category', 'gender', 'state']]

scale before test


2. Transform
3. Test
4. Generate
5. Test
6. Select
7. Test, test learning
8. Imbalance
9. Test, test learning

### Feature generation

In [13]:
pf = PolynomialFeatures(degree=4, include_bias=False)
X_train_num = pd.DataFrame(pf.fit_transform(X_train_num), columns=pf.get_feature_names_out(X_train_num.columns))
X_train_num

,time,amount,city_pop,industry_code,age,time^2,time amount,time city_pop,time industry_code,time age,amount^2,amount city_pop,amount industry_code,amount age,city_pop^2,city_pop industry_code,city_pop age,industry_code^2,industry_code age,age^2,time^3,time^2 amount,time^2 city_pop,time^2 industry_code,time^2 age,time amount^2,time amount city_pop,time amount industry_code,time amount age,time city_pop^2,time city_pop industry_code,time city_pop age,time industry_code^2,time industry_code age,time age^2,amount^3,amount^2 city_pop,amount^2 industry_code,amount^2 age,amount city_pop^2,amount city_pop industry_code,amount city_pop age,amount industry_code^2,amount industry_code age,amount age^2,city_pop^3,city_pop^2 industry_code,city_pop^2 age,city_pop industry_code^2,city_pop industry_code age,city_pop age^2,industry_code^3,industry_code^2 age,industry_code age^2,age^3,time^4,time^3 amount,time^3 city_pop,time^3 industry_code,time^3 age,time^2 amount^2,time^2 amount city_pop,time^2 amount industry_code,time^2 amount age,time^2 city_pop^2,time^2 city_pop industry_code,time^2 city_pop age,time^2 industry_code^2,time^2 industry_code age,time^2 age^2,time amount^3,time amount^2 city_pop,time amount^2 industry_code,time amount^2 age,time amount city_pop^2,time amount city_pop industry_code,time amount city_pop age,time amount industry_code^2,time amount industry_code age,time amount age^2,time city_pop^3,time city_pop^2 industry_code,time city_pop^2 age,time city_pop industry_code^2,time city_pop industry_code age,time city_pop age^2,time industry_code^3,time industry_code^2 age,time industry_code age^2,time age^3,amount^4,amount^3 city_pop,amount^3 industry_code,amount^3 age,amount^2 city_pop^2,amount^2 city_pop industry_code,amount^2 city_pop age,amount^2 industry_code^2,amount^2 industry_code age,amount^2 age^2,amount city_pop^3,amount city_pop^2 industry_code,amount city_pop^2 age,amount city_pop industry_code^2,amount city_pop industry_code age,amount city_pop age^2,amount industry_code^3,amount industry_code^2 age,amount industry_code age^2,amount age^3,city_pop^4,city_pop^3 industry_code,city_pop^3 age,city_pop^2 industry_code^2,city_pop^2 industry_code age,city_pop^2 age^2,city_pop industry_code^3,city_pop industry_code^2 age,city_pop industry_code age^2,city_pop age^3,industry_code^4,industry_code^3 age,industry_code^2 age^2,industry_code age^3,age^4
0,1.553916e+09,61.56,2135.0,4.0,20.501027,2.414656e+18,9.565910e+10,3.317612e+12,6.215666e+09,3.185688e+10,3789.6336,131430.60,246.24,1262.043203,4.558225e+06,8540.0,4.376969e+04,16.0,82.004107,420.292096,3.752174e+27,1.486462e+20,5.155291e+21,9.658625e+18,4.950293e+19,5.888774e+12,2.042322e+14,3.826364e+11,1.961110e+12,7.083101e+15,1.327045e+13,6.801444e+13,2.486266e+10,1.274275e+11,6.530988e+11,2.332898e+05,8.090868e+06,15158.5344,7.769138e+04,2.806043e+08,5.257224e+05,2.694462e+06,984.96,5048.172813,25873.181400,9.731810e+09,1.823290e+07,9.344829e+07,34160.0,1.750788e+05,8.973236e+05,64.0,328.016427,1681.168382,8616.419469,5.830565e+36,2.309838e+29,8.010892e+30,1.500870e+28,7.692342e+28,9.150663e+21,3.173597e+23,5.945850e+20,3.047401e+21,1.100655e+25,2.062116e+22,1.056888e+23,3.863450e+19,1.980117e+20,1.014861e+21,3.625129e+14,1.257253e+16,2.355510e+13,1.207259e+14,4.360357e+17,8.169287e+14,4.186969e+15,1.530546e+12,7.844439e+12,4.020476e+13,1.512242e+19,2.833240e+16,1.452108e+17,5.308179e+13,2.720578e+14,1.394366e+15,9.945065e+10,5.097101e+11,2.612395e+12,1.338920e+13,1.436132e+07,4.980738e+08,9.331594e+05,4.782681e+06,1.727400e+10,3.236347e+07,1.658711e+08,60634.1376,3.107655e+05,1.592753e+06,5.990902e+11,1.122417e+09,5.752677e+09,2.102890e+06,1.077785e+07,5.523924e+07,3939.84,20192.691253,1.034927e+05,5.304268e+05,2.077742e+13,3.892724e+10,1.995121e+11,7.293160e+07,3.737932e+08,1.915786e+09,136640.0,7.003151e+05,3.589294e+06,1.839606e+07,256.0,1312.065708,6724.673528,3.446568e+04,1.766454e+05
1,1.576587e+09,47.43,1263321.0,4.0,65.363450,2.485628e+18,7.477754e+10,1.991736e+

In [15]:
import sys
print(sys.getsizeof(X_train_num))

972506164


In [23]:
def evaluate(X, y):
    models = dict()
    
    models['Logistic Regression'] = LogisticRegression(random_state=1, max_iter=100000, class_weight='balanced')    
    #models['SVC'] = SVC(class_weight='balanced', random_state=1)
    models['SGD'] = SGDClassifier(random_state=1, class_weight='balanced')
    models['Nearest Neighbor'] = KNeighborsClassifier(3)
    models['Naive Bayes'] = GaussianNB()
    models['Decision Tree'] = DecisionTreeClassifier(random_state=1, class_weight='balanced')
    models['Random Forest'] = RandomForestClassifier(random_state=1, class_weight='balanced')
    models['MLPC'] = MLPClassifier(random_state=1)

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=1)
    
    names, res = [], []

    for name, model in models.items():
        print('         ', name)
        scores = cross_val_score(model, X, y, scoring='f1_score', cv=cv, n_jobs=10)#scoring='accuracy'
        print('Min:', min(scores), 'Max:', max(scores), 'Mean:', np.mean(scores))
        res.append(scores)
        names.append(name)

    plt.figure(figsize=(10, 8))
    plt.boxplot(res, labels=names, showmeans=True)
    plt.grid()
    plt.xticks(rotation=25);

### Feature selection

In [17]:
X_train_num = pd.DataFrame(StandardScaler().fit_transform(X_train_num), columns=X_train_num.columns)

In [19]:
enc = OneHotEncoder(drop='first', sparse_output=False, dtype=bool)
X_train_cat = pd.DataFrame(enc.fit_transform(X_train_cat), columns=enc.get_feature_names_out())

In [21]:
X_train = pd.concat([X_train_num, X_train_cat], axis=1)

In [22]:
X_train

,time,amount,city_pop,industry_code,age,time^2,time amount,time city_pop,time industry_code,time age,amount^2,amount city_pop,amount industry_code,amount age,city_pop^2,city_pop industry_code,city_pop age,industry_code^2,industry_code age,age^2,time^3,time^2 amount,time^2 city_pop,time^2 industry_code,time^2 age,time amount^2,time amount city_pop,time amount industry_code,time amount age,time city_pop^2,time city_pop industry_code,time city_pop age,time industry_code^2,time industry_code age,time age^2,amount^3,amount^2 city_pop,amount^2 industry_code,amount^2 age,amount city_pop^2,amount city_pop industry_code,amount city_pop age,amount industry_code^2,amount industry_code age,amount age^2,city_pop^3,city_pop^2 industry_code,city_pop^2 age,city_pop industry_code^2,city_pop industry_code age,city_pop age^2,industry_code^3,industry_code^2 age,industry_code age^2,age^3,time^4,time^3 amount,time^3 city_pop,time^3 industry_code,time^3 age,time^2 amount^2,time^2 amount city_pop,time^2 amount industry_code,time^2 amount age,time^2 city_pop^2,time^2 city_pop industry_code,time^2 city_pop age,time^2 industry_code^2,time^2 industry_code age,time^2 age^2,time amount^3,time amount^2 city_pop,time amount^2 industry_code,time amount^2 age,time amount city_pop^2,time amount city_pop industry_code,time amount city_pop age,time amount industry_code^2,time amount industry_code age,time amount age^2,time city_pop^3,time city_pop^2 industry_code,time city_pop^2 age,time city_pop industry_code^2,time city_pop industry_code age,time city_pop age^2,time industry_code^3,time industry_code^2 age,time industry_code age^2,time age^3,amount^4,amount^3 city_pop,amount^3 industry_code,amount^3 age,amount^2 city_pop^2,amount^2 city_pop industry_code,amount^2 city_pop age,amount^2 industry_code^2,amount^2 industry_code age,amount^2 age^2,amount city_pop^3,amount city_pop^2 industry_code,amount city_pop^2 age,amount city_pop industry_code^2,amount city_pop industry_code age,amount city_pop age^2,amount industry_code^3,amount industry_code^2 age,amount industry_code age^2,amount age^3,city_pop^4,city_pop^3 industry_code,city_pop^3 age,city_pop^2 industry_code^2,city_pop^2 industry_code age,city_pop^2 age^2,city_pop industry_code^3,city_pop industry_code^2 age,city_pop industry_code age^2,city_pop age^3,industry_code^4,industry_code^3 age,industry_code^2 age^2,industry_code age^3,age^4,category_food_dining,category_gas_transport,category_grocery_net,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel,gender_M,state_AL,state_AR,state_AZ,state_CA,state_CO,state_CT,state_DC,state_DE,state_FL,state_GA,state_HI,state_IA,state_ID,state_IL,state_IN,state_KS,state_KY,state_LA,state_MA,state_MD,state_ME,state_MI,state_MN,state_MO,state_MS,state_MT,state_NC,state_ND,state_NE,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OH,state_OK,state_OR,state_PA,state_RI,state_SC,state_SD,state_TN,state_TX,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY
0,-1.259216,-0.054417,-0.287405,0.305652,-1.758199,-1.256755,-0.058295,-0.287477,0.271422,-1.769877,-0.014371,-0.109573,-0.013482,-0.250204,-0.159566,-0.264749,-0.297210,0.142090,-1.143290,-1.248641,-1.254265,-0.062148,-0.287529,0.237188,-1.780357,-0.014362,-0.109515,-0.017295,-0.251296,-0.159569,-0.264830,-0.297245,0.124828,-1.152427,-1.250697,-0.004288,-0.007293,-0.011525,-0.012702,-0.067848,-0.110532,-0.115024,-0.013604,-0.208525,-0.275079,-0.112223,-0.140531,-0.166813,-0.231840,-0.264230,-0.282982,-0.007271,-0.741264,-1.064964,-0.915204,-1.251748,-0.065976,-0.287559,0.203023,-1.789623,-0.014352,-0.109455,-0.021082,-0.252362,-0.159561,-0.264891,-0.297257,0.107614,-1.161121,-1.252481,-0.004282,-0.007280,-0.011506,-0.012670,-0.067805,-0.110489,-0.114928,-0.016729,-0.209496,-0.275073,-0.112226,-0.140537,-0.166819,-0.231911,-0.264264,-0.282992,-0.017697,-0.747141,-1.066934

In [25]:
pca = PCA(n_components=0.99, copy=False)
X_train_num = pd.DataFrame(pca.fit_transform(X_train_num))
print(X_train_num.shape)
X_train_num.info()

(972506, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 972506 entries, 0 to 972505
Data columns (total 18 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   0       972506 non-null  float64
 1   1       972506 non-null  float64
 2   2       972506 non-null  float64
 3   3       972506 non-null  float64
 4   4       972506 non-null  float64
 5   5       972506 non-null  float64
 6   6       972506 non-null  float64
 7   7       972506 non-null  float64
 8   8       972506 non-null  float64
 9   9       972506 non-null  float64
 10  10      972506 non-null  float64
 11  11      972506 non-null  float64
 12  12      972506 non-null  float64
 13  13      972506 non-null  float64
 14  14      972506 non-null  float64
 15  15      972506 non-null  float64
 16  16      972506 non-null  float64
 17  17      972506 non-null  float64
dtypes: float64(18)
memory usage: 133.6 MB


In [ ]:
ct = ColumnTransformer(
    [
        ("text_preprocess", OneHotEncoder(drop='first', sparse_output=False, dtype=bool), ['category', 'gender', 'state'])
    ],
    remainder='passthrough', verbose_feature_names_out=False).set_output(transform='pandas') 

### Handling imbalanced data